# Dollar-Cost Averaging vs Lump-Sum Investing — France

Backtests LSI vs DCA on Euronext Paris tickers over rolling 5-year windows. All simulation, metrics and plotting logic lives in the `lsi_dca` package next to this notebook — this notebook only orchestrates and displays.

**Fixed vs the original script**: prices are fetched raw (`auto_adjust=False`) so dividends aren't double-counted (once via price adjustment, once via the explicit DRIP loop), the LSI equity curve is now built day-by-day (needed for correct Sharpe/Sortino), and idle DCA cash earns the historical Livret A rate instead of 0%.

In [ ]:
import logging
import sys

sys.path.insert(0, '.')  # lsi_dca package lives alongside this notebook

import lsi_dca
from lsi_dca import plotting

lsi_dca.configure_logging(logging.INFO)   # one curated line per run; use logging.DEBUG for a per-window trace
%matplotlib inline

In [ ]:
import pandas as pd

## Table 1 — CAC 40 index, three DCA frequencies

$W_0 = 12{,}000$€, $t_0=$ 2010-01-04, 5-year rolling windows, 60 windows, 1-month shift.

In [ ]:
COMMON = dict(w0=12_000, t0="2010-01-04", window_years=5, shift_months=1, n_windows=60, rf_annual=0.02)

table1 = lsi_dca.backtest_frequencies("^FCHI", **COMMON, freqs=("M", "S", "A"))
table1

## A closer look — one representative window (CAC 40, semi-annual DCA)

The five charts below are chosen to answer five distinct questions: what does the wealth path look like, how spread out are outcomes, does LSI's edge depend on entry timing, does the market regime matter, and what's the risk/return trade-off.

In [ ]:
report = lsi_dca.compare("^FCHI", **COMMON, dca_freq="S")
report.summary

In [ ]:
first_t0, first_tf = report.results['t0'].iloc[0], report.results['tf'].iloc[0]
lsi = lsi_dca.simulate_lsi("^FCHI", 12_000, first_t0, first_tf)
dca = lsi_dca.simulate_dca("^FCHI", 12_000, first_t0, first_tf, freq="S")

plotting.equity_curves(lsi.equity_curve, dca.equity_curve, title="CAC 40 — portfolio value, first window")

In [ ]:
plotting.hpr_distribution(report.results, ticker="CAC 40")

In [ ]:
plotting.hpr_gap_timeline(report.results, ticker="CAC 40")

In [ ]:
plotting.win_rate_by_regime(report.regimes, ticker="CAC 40")

In [ ]:
plotting.risk_return(report.results, ticker="CAC 40")

## Statistical significance

Paired tests (same market path per window for both strategies -> common cross-sectional variance cancels out).

In [ ]:
report.tests

## Table 2 — Cross-section, 37 CAC 40 constituents (semi-annual DCA)

In [ ]:
CONSTITUENTS = {
    "Air Liquide": "AI.PA", "Airbus": "AIR.PA", "AXA": "CS.PA", "BNP Paribas": "BNP.PA",
    "Bouygues": "EN.PA", "Capgemini": "CAP.PA", "Carrefour": "CA.PA", "Credit Agricole": "ACA.PA",
    "Danone": "BN.PA", "Dassault Systemes": "DSY.PA", "Engie": "ENGI.PA", "EssilorLuxottica": "EL.PA",
    "Eurofins": "ERF.PA", "Hermes": "RMS.PA", "Kering": "KER.PA", "Legrand": "LR.PA",
    "L'Oreal": "OR.PA", "LVMH": "MC.PA", "Michelin": "ML.PA", "Orange": "ORA.PA",
    "Pernod Ricard": "RI.PA", "Publicis": "PUB.PA", "Renault": "RNO.PA", "Safran": "SAF.PA",
    "Saint-Gobain": "SGO.PA", "Sanofi": "SAN.PA", "Schneider Electric": "SU.PA",
    "Societe Generale": "GLE.PA", "Stellantis": "STLAM.MI", "Teleperformance": "TEP.PA",
    "Thales": "HO.PA", "TotalEnergies": "TTE.PA", "Veolia": "VIE.PA", "Vinci": "DG.PA",
    "Vivendi": "VIV.PA", "Accor": "AC.PA", "Christian Dior": "CDI.PA",
}

rows = []
for name, ticker in CONSTITUENTS.items():
    try:
        r = lsi_dca.compare(ticker, **COMMON, dca_freq="S")
        rows.append({
            "Company": name, "Ticker": ticker,
            "LSI mean HPR (%)": r.summary.loc["HPR (%)", "lsi_mean"],
            "DCA mean HPR (%)": r.summary.loc["HPR (%)", "dca_mean"],
            "LSI win rate (%)": r.lsi_win_rate,
        })
    except Exception as e:
        print(f"skipped {name} ({ticker}): {e}")

table2 = pd.DataFrame(rows).set_index("Company").sort_values("LSI win rate (%)")
table2

## Robustness — idle DCA cash at 0% vs the Livret A schedule

Re-runs the cross-section with `cash_rf_annual=0.0` (the original paper's assumption) to quantify how much of DCA's shortfall was simply un-remunerated cash.

In [ ]:
robust_rows = []
for name, ticker in CONSTITUENTS.items():
    try:
        no_yield = lsi_dca.compare(ticker, **COMMON, dca_freq="S", cash_rf_annual=0.0)
        with_yield = lsi_dca.compare(ticker, **COMMON, dca_freq="S", cash_rf_annual=None)
        robust_rows.append({
            "Company": name, "Ticker": ticker,
            "DCA mean HPR before (%)": no_yield.summary.loc["HPR (%)", "dca_mean"],
            "DCA mean HPR after (%)": with_yield.summary.loc["HPR (%)", "dca_mean"],
            "LSI win rate before (%)": no_yield.lsi_win_rate,
            "LSI win rate after (%)": with_yield.lsi_win_rate,
            "Win rate change (pts)": with_yield.lsi_win_rate - no_yield.lsi_win_rate,
        })
    except Exception as e:
        print(f"skipped {name} ({ticker}): {e}")

robustness = pd.DataFrame(robust_rows).set_index("Company").sort_values("Win rate change (pts)")
robustness

## Export — consolidated Excel workbook

In [ ]:
with pd.ExcelWriter("comparaison_finale_corrigee.xlsx", engine="openpyxl") as writer:
    table1.to_excel(writer, sheet_name="Table 1 - CAC40 Index")
    table2.to_excel(writer, sheet_name="Table 2 - Cross Section")
    robustness.to_excel(writer, sheet_name="Cash Yield Robustness")
print("Exported -> comparaison_finale_corrigee.xlsx")